# OCT Eye Disease – EfficientNet-B1 Complete Testing Notebook

This notebook evaluates the fine-tuned **EfficientNet-B1** transfer-learning model (`efficientnet_b1_final.keras`) end-to-end.

**Key parameters – must match training notebooks exactly:**

| Parameter | Value |
|---|---|
| Image size | 224 × 224 |
| Color mode | grayscale (1 channel) → replicated to 3 channels for EfficientNet |
| Normalisation | `efficientnet.preprocess_input()` **inside** the model — do **NOT** divide manually |
| Label mode | categorical (one-hot) |
| Class order | alphabetical: `CNV, DME, DRUSEN, NORMAL` |
| Mixed precision | `mixed_float16` (same as training) |

**Sections**
1. Mount Google Drive & Load Model  
2. Imports & Configuration  
3. Load Datasets  
4. Quick Sanity Check  
5. Medical Diagnostic Grid – 5 × 4 Sample Visualisation  
6. Full Evaluation – Confusion Matrix & Classification Report  
7. Advanced Metrics (Balanced Accuracy, MCC, Cohen's κ)  
8. ROC Curves (One-vs-Rest)  
9. Precision-Recall Curves  
10. Per-Split Metrics Summary  
11. Training History Visualisation (Stage 1 + Stage 2)  
12. Save Outputs to Google Drive  


## 1 · Mount Google Drive & Load Model

In [1]:
# Optional Colab / Kaggle setup.
# Uncomment only when running in a fresh notebook environment.

import os
os.makedirs("/root/.kaggle", exist_ok=True)
!pip install kaggle==1.6.17
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d anirudhcv/labeled-optical-coherence-tomography-oct
!unzip /content/labeled-optical-coherence-tomography-oct.zip -d /content/


Streaming output truncated to the last 5000 lines.
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5822041-15.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5822041-16.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5822041-18.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5822041-2.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5822041-8.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-582215-6.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-582215-7.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-582215-8.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5823731-1.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5823731-5.jpeg  
  inflating: /content/Dataset - train+val+test/val/NORMAL/NORMAL-5824080-5.jpeg  
  inflating: /content/Dataset - train+val+test/

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import zipfile, json, shutil

MODEL_PATH = "/content/drive/MyDrive/oct_outputs_efficientnet_b1/efficientnet_b1_final.keras"
PATCHED_PATH = "/content/efficientnet_b1_final_patched.keras"

shutil.copy(MODEL_PATH, PATCHED_PATH)

# Read config.json out of the archive
with zipfile.ZipFile(PATCHED_PATH, 'r') as z:
    config = json.loads(z.read('config.json'))

# Find the Lambda layer(s) and inspect them
def find_lambdas(layer_config, path=""):
    if isinstance(layer_config, dict):
        if layer_config.get("class_name") == "Lambda":
            print(path, "->", layer_config["config"].get("name"))
        for k, v in layer_config.items():
            find_lambdas(v, path + "/" + k)
    elif isinstance(layer_config, list):
        for i, v in enumerate(layer_config):
            find_lambdas(v, path + f"[{i}]")

find_lambdas(config)

/config/layers[1] -> lambda


In [4]:
import zipfile, json

with zipfile.ZipFile(MODEL_PATH, 'r') as z:
    config = json.loads(z.read('config.json'))

lambda_layer = config["config"]["layers"][1]
print(json.dumps(lambda_layer, indent=2))

{
  "module": "keras.layers",
  "class_name": "Lambda",
  "config": {
    "name": "lambda",
    "trainable": true,
    "dtype": {
      "module": "keras",
      "class_name": "DTypePolicy",
      "config": {
        "name": "float32"
      },
      "registered_name": null
    },
    "function": {
      "class_name": "__lambda__",
      "config": {
        "code": "4wEAAAAAAAAAAAAAAAUAAAADAAAA8zYAAACXAHQAAAAAAAAAAABqAwAAAAAAAAAAAAAAAAAAAAAA\nAHwAfAB8AGcDZAGsAqsCAAAAAAAAUwApA07p/////ykB2gRheGlzKQLaAnRm2gZjb25jYXQpAdoD\naW1ncwEAAAAg+iEvdG1wL2lweWtlcm5lbF81OTgyLzM1NzA2NzExMzEucHnaCDxsYW1iZGE+cggA\nAAAQAAAAcxcAAACAAKRyp3mhebAjsHO4Q7AfwHKgedQnSvMAAAAA\n",
        "defaults": null,
        "closure": null
      }
    },
    "arguments": {}
  },
  "registered_name": null,
  "build_config": {
    "input_shape": [
      null,
      224,
      224,
      1
    ]
  },
  "name": "lambda",
  "inbound_nodes": [
    {
      "args": [
        {
          "class_name": "__keras_tensor__",
          "con

In [5]:
import zipfile, json, shutil, os

shutil.copy(MODEL_PATH, PATCHED_PATH)

# Read all files from the original archive
with zipfile.ZipFile(PATCHED_PATH, 'r') as z:
    names = z.namelist()
    contents = {name: z.read(name) for name in names}

# Modify config.json
config = json.loads(contents['config.json'])
lambda_layer = config["config"]["layers"][1]["config"]
lambda_layer["output_shape"] = [224, 224, 3]
contents['config.json'] = json.dumps(config).encode('utf-8')

# Rewrite the archive with the patched config.json, everything else unchanged
os.remove(PATCHED_PATH)
with zipfile.ZipFile(PATCHED_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for name, data in contents.items():
        z.writestr(name, data)

print("Patched model saved to:", PATCHED_PATH)

MODEL_PATH = PATCHED_PATH

Patched model saved to: /content/efficientnet_b1_final_patched.keras


In [6]:
import tensorflow as tf
from tensorflow.keras import mixed_precision

# Enable the same mixed precision as training
mixed_precision.set_global_policy('mixed_float16')

model = tf.keras.models.load_model(MODEL_PATH, safe_mode=False)
model.summary()
print("\nModel loaded successfully.")
print("Input shape expected by model:", model.input_shape)


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 332 variables whereas the saved optimizer has 328 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 328 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "efficientnet_b1_oct"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ oct_input (InputLayer)          │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb1 (Functional)     │ (None, 7, 7, 1280)     │     6,575,239 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,180,617 (80.80 MB)

 Trainable params: 6,907,036 (26.35 MB)

 Non-trainable params: 459,503 (1.75 MB)

 Optimizer params: 13,814,078 (52.70 MB)


Model loaded successfully.
Input shape expected by model: (None, 224, 224, 1)


## 2 · Imports & Configuration

In [7]:
import os
import json
import gc
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    roc_curve, auc, roc_auc_score, precision_recall_curve,
)
from sklearn.preprocessing import label_binarize

sns.set_theme(style="whitegrid")

# ── Must match training notebooks exactly ─────────────────────────────────────
IMG_SIZE     = (224, 224)      # EfficientNet-B1 trained at 224×224
COLOR_MODE   = "grayscale"     # grayscale → replicated to 3-ch inside model
NUM_CHANNELS = 1
BATCH_SIZE   = 16
SEED         = 42
AUTOTUNE     = tf.data.AUTOTUNE

DATASET_ROOT = "/content/Dataset - train+val+test"
TRAIN_DIR    = os.path.join(DATASET_ROOT, "train")
VAL_DIR      = os.path.join(DATASET_ROOT, "val")
TEST_DIR     = os.path.join(DATASET_ROOT, "test")

# Output dir (Google Drive) – mirrors training notebooks
OUTPUT_DIR = Path("/content/drive/MyDrive/oct_outputs_efficientnet_b1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow version:", tf.__version__)
print(f"Image size: {IMG_SIZE}")
print(f"Color mode: {COLOR_MODE}")
print(f"Output directory: {OUTPUT_DIR}")


TensorFlow version: 2.20.0
Image size: (224, 224)
Color mode: grayscale
Output directory: /content/drive/MyDrive/oct_outputs_efficientnet_b1


## 3 · Load Datasets

Datasets are loaded with **exactly the same parameters as in training**:
- `color_mode="grayscale"` → tensors have shape `(batch, 224, 224, 1)`
- `label_mode="categorical"` → one-hot labels
- `shuffle=False` for val/test so ground-truth order is preserved
- **No manual rescaling** — `efficientnet.preprocess_input()` is baked into the model graph via the Concatenate + preprocess_input layers


In [8]:
def load_dataset(split_dir, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        split_dir,
        labels="inferred",
        label_mode="categorical",
        color_mode=COLOR_MODE,       # grayscale → (H, W, 1)
        batch_size=BATCH_SIZE,
        image_size=IMG_SIZE,         # 224×224
        shuffle=shuffle,
        seed=SEED if shuffle else None,
        interpolation="bilinear",
    )
    # Read class_names BEFORE .prefetch() — PrefetchDataset hides the attribute
    class_names = ds.class_names
    return ds.prefetch(AUTOTUNE), class_names


training_set, class_names = load_dataset(TRAIN_DIR, shuffle=True)
validation_set, _         = load_dataset(VAL_DIR,   shuffle=False)
test_set, _               = load_dataset(TEST_DIR,  shuffle=False)

num_classes = len(class_names)

print("Classes (alphabetical order):", class_names)
print("Number of classes:", num_classes)
print("Training element spec:", training_set.element_spec)


Found 76515 files belonging to 4 classes.
Found 21861 files belonging to 4 classes.
Found 10933 files belonging to 4 classes.
Classes (alphabetical order): ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Number of classes: 4
Training element spec: (TensorSpec(shape=(None, 224, 224, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 4), dtype=tf.float32, name=None))


## 4 · Quick Sanity Check

In [9]:
sample_images, sample_labels = next(iter(test_set))

print("Batch images shape:", sample_images.shape)   # (16, 224, 224, 1)
print("Batch labels shape:", sample_labels.shape)   # (16, 4)
print("Pixel dtype:", sample_images.dtype)
print("Raw pixel range: [{:.0f}, {:.0f}]".format(
    float(tf.reduce_min(sample_images)),
    float(tf.reduce_max(sample_images)),
))
print()

# Patch the Lambda layer's function to have `tf` properly in scope
lambda_layer = model.get_layer('lambda')
lambda_layer.function = lambda imgs: tf.concat([imgs, imgs, imgs], axis=-1)

# Now test
sample_images, sample_labels = next(iter(test_set))
dummy_out = model(sample_images, training=False)
print("Model output shape:", dummy_out.shape)        # (16, 4)
print("Prob sum per sample (should be ~1.0):",
      np.round(np.sum(dummy_out.numpy(), axis=1), 4))


Batch images shape: (16, 224, 224, 1)
Batch labels shape: (16, 4)
Pixel dtype: <dtype: 'float32'>
Raw pixel range: [0, 255]

Model output shape: (16, 4)
Prob sum per sample (should be ~1.0): [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


## 5 · Medical Diagnostic Grid – 5 × 4 Sample Visualisation

20 representative OCT images (5 per class) in a clinical-style grid.

Each panel shows:
- **True-label badge** (top-left) and **sample index** (top-right)
- **Prediction verdict** (bottom overlay — ✓ correct, ✗ wrong) with confidence %
- **Per-class probability bars** flush to image width, zero gap between image and bars


In [10]:
import glob, random

random.seed(32)
N_PER_CLASS = 5

def sample_files(cls, n):
    all_files = sorted(
        glob.glob(f"{TEST_DIR}/{cls}/*.jpeg") +
        glob.glob(f"{TEST_DIR}/{cls}/*.jpg")  +
        glob.glob(f"{TEST_DIR}/{cls}/*.png")
    )
    if len(all_files) < n:
        raise FileNotFoundError(f"Only {len(all_files)} files for {cls}, need {n}")
    return random.sample(all_files, n)

SAMPLE_PATHS = []
TRUE_LABELS  = []
for cls in class_names:
    files = sample_files(cls, N_PER_CLASS)
    SAMPLE_PATHS += files
    TRUE_LABELS  += [cls] * N_PER_CLASS

print(f"Loaded {len(SAMPLE_PATHS)} sample paths")
print("Label distribution:", {c: TRUE_LABELS.count(c) for c in class_names})

# ── Colour palette ─────────────────────────────────────────────────────────────
CLASS_COLORS = {
    "CNV":    "#C0392B",
    "DME":    "#2980B9",
    "DRUSEN": "#D4AC0D",
    "NORMAL": "#1A9641",
}
BG_COLOR    = "#1A1F2E"
PANEL_COLOR = "#F4F6FA"
TEXT_COLOR  = "#FFFFFF"
MUTED_COLOR = "#8892A4"
CORRECT_CLR = "#1A9641"
WRONG_CLR   = "#C0392B"


def predict_image_grid(path):
    img_gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise FileNotFoundError(f"Could not read: {path}")
    img_resized = cv2.resize(img_gray, IMG_SIZE)
    # Model expects (batch, H, W, 1) — no manual normalisation
    inp = img_resized[np.newaxis, :, :, np.newaxis].astype(np.float32)
    probs = model.predict(inp, verbose=0)[0]
    return img_resized, probs, class_names[int(np.argmax(probs))]


results = [predict_image_grid(p) for p in SAMPLE_PATHS]

# ── Figure ────────────────────────────────────────────────────────────────────
ROWS, COLS = N_PER_CLASS, len(class_names)  # 5 × 4 = 20
N          = ROWS * COLS

fig = plt.figure(figsize=(16, 22), facecolor=BG_COLOR)
fig.patch.set_facecolor(BG_COLOR)

# Dot-grid background
_bg = fig.add_axes([0, 0, 1, 1], zorder=0)
_bg.set_xlim(0, 1); _bg.set_ylim(0, 1)
_bg.axis("off"); _bg.set_facecolor(BG_COLOR)
_sp = 0.005
_gx, _gy = np.meshgrid(np.arange(_sp, 1.0, _sp), np.arange(_sp, 1.0, _sp))
_bg.scatter(_gx.ravel(), _gy.ravel(), s=1.6, color="#C6D2E8",
            alpha=0.70, zorder=0, linewidths=0)

# Header
fig.text(0.5, 0.993,
         "RETINAL OCT  ·  EfficientNet-B1 Transfer Learning  ·  DIAGNOSTIC REVIEW",
         ha="center", va="top", fontsize=13, fontweight="bold",
         color=TEXT_COLOR, fontfamily="monospace")
fig.text(0.5, 0.983,
         f"efficientnet_b1_final.keras   |   CNV · DME · DRUSEN · NORMAL   |   n = {N}",
         ha="center", va="top", fontsize=8.5, color=MUTED_COLOR,
         fontfamily="monospace")

correct_all = sum(1 for i in range(N) if results[i][2] == TRUE_LABELS[i])
acc_col = CORRECT_CLR if correct_all / N >= 0.8 else WRONG_CLR
fig.text(0.5, 0.974,
         f"Grid accuracy: {correct_all}/{N}  ({correct_all/N*100:.0f}%)",
         ha="center", va="top", fontsize=8.5, color=acc_col,
         fontfamily="monospace", fontweight="bold")

# Legend
leg = fig.add_axes([0.04, 0.959, 0.92, 0.009])
leg.set_xlim(0, 1); leg.set_ylim(0, 1)
leg.axis("off"); leg.patch.set_alpha(0)
for i, cn in enumerate(class_names):
    x = 0.06 + i * 0.185
    leg.add_patch(FancyBboxPatch((x, 0.05), 0.065, 0.9,
                                  boxstyle="round,pad=0.05",
                                  facecolor=CLASS_COLORS[cn], edgecolor="none", alpha=0.9))
    leg.text(x + 0.0325, 0.52, cn, ha="center", va="center",
             fontsize=8, fontweight="bold", color="white", fontfamily="monospace")
for j, (lbl, clr) in enumerate([("✓ correct", CORRECT_CLR), ("✗ wrong", WRONG_CLR)]):
    x = 0.74 + j * 0.13
    leg.add_patch(FancyBboxPatch((x, 0.05), 0.105, 0.9,
                                  boxstyle="round,pad=0.05",
                                  facecolor=clr, edgecolor="none", alpha=0.18))
    leg.text(x + 0.0525, 0.52, lbl, ha="center", va="center",
             fontsize=7.5, color=clr, fontfamily="monospace")

# Main grid
outer = gridspec.GridSpec(
    ROWS, COLS, figure=fig,
    left=0.04, right=0.96, top=0.953, bottom=0.012,
    hspace=0.07, wspace=0.06,
)

for idx in range(N):
    row, col = divmod(idx, COLS)
    img_gray, probs, pred_label = results[idx]
    true_label = TRUE_LABELS[idx]
    is_correct = pred_label == true_label
    accent     = CORRECT_CLR if is_correct else WRONG_CLR
    check      = "✓" if is_correct else "✗"
    pred_conf  = probs[class_names.index(pred_label)] * 100

    inner = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=outer[row, col],
        height_ratios=[9, 1],
        hspace=0,
    )

    ax_img = fig.add_subplot(inner[0])
    ax_img.imshow(img_gray, cmap="bone", vmin=0, vmax=255, aspect="auto")
    ax_img.set_xticks([]); ax_img.set_yticks([])
    for side, sp in ax_img.spines.items():
        sp.set_visible(side != "bottom")
        if side != "bottom":
            sp.set_edgecolor(accent); sp.set_linewidth(1.8)
    ax_img.set_facecolor(PANEL_COLOR)

    ax_img.text(0.025, 0.965, f" {true_label} ",
                transform=ax_img.transAxes, ha="left", va="top",
                fontsize=8, fontweight="bold", color="white", fontfamily="monospace",
                bbox=dict(facecolor=CLASS_COLORS[true_label],
                          edgecolor="none", boxstyle="round,pad=0.18", alpha=0.93))
    ax_img.text(0.975, 0.965, f"#{idx+1:02d}",
                transform=ax_img.transAxes, ha="right", va="top",
                fontsize=7, color=MUTED_COLOR, fontfamily="monospace",
                bbox=dict(facecolor="#E8ECF3", edgecolor="none",
                          boxstyle="round,pad=0.12", alpha=0.90))
    ax_img.text(0.5, 0.04,
                f"{check}  {pred_label}  {pred_conf:.0f}%",
                transform=ax_img.transAxes, ha="center", va="bottom",
                fontsize=8, fontweight="bold", color="white", fontfamily="monospace",
                bbox=dict(facecolor=accent, edgecolor="none",
                          boxstyle="round,pad=0.22", alpha=0.88))

    ax_bar = fig.add_subplot(inner[1])
    ax_bar.set_facecolor(PANEL_COLOR)
    for side, sp in ax_bar.spines.items():
        sp.set_visible(side != "top")
        if side != "top":
            sp.set_edgecolor(accent); sp.set_linewidth(1.8)
    ax_bar.set_xticks([]); ax_bar.set_yticks([])

    x_cursor = 0.0
    for cn in class_names:
        p = float(probs[class_names.index(cn)])
        ax_bar.barh(0, p * 100, left=x_cursor * 100,
                    color=CLASS_COLORS[cn], height=1.0, edgecolor="none",
                    alpha=0.85 if cn == pred_label else 0.38)
        x_cursor += p

    cumulative = 0.0
    for cn in class_names[:-1]:
        cumulative += float(probs[class_names.index(cn)])
        ax_bar.axvline(x=cumulative * 100, color="white", linewidth=0.8, alpha=0.6)

    ax_bar.set_xlim(0, 100)
    ax_bar.set_ylim(-0.5, 0.5)

grid_path = str(OUTPUT_DIR / "efficientnet_diagnostic_grid.png")
plt.savefig(grid_path, dpi=300, bbox_inches="tight",
            facecolor=BG_COLOR, edgecolor="none")
plt.show()
print(f"Grid saved → {grid_path}  |  Accuracy: {correct_all}/{N} ({correct_all/N*100:.0f}%)")


Output hidden; open in https://colab.research.google.com to view.

In [11]:
# # Uncomment to save outputs to Google Drive
# OUTPUT_DIR = OUTPUT_DIR / "test"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# cm_df.to_csv(OUTPUT_DIR / "confusion_matrix.csv")
# adv_df.to_csv(OUTPUT_DIR / "advanced_metrics_per_class.csv", index=False)
# metrics_df.to_csv(OUTPUT_DIR / "split_metrics.csv", index=False)
# np.save(str(OUTPUT_DIR / "y_pred_proba.npy"), y_pred_proba)
# np.save(str(OUTPUT_DIR / "y_true.npy"), y_true)
# np.save(str(OUTPUT_DIR / "y_pred.npy"), y_pred)
# print("Results saved to", OUTPUT_DIR)
# print("Uncomment the block above to save results to Google Drive.")
